📌 Description du script :
Ce script permet de préparer et augmenter un dataset d’images pour atteindre exactement 2000 images, tout en conservant les caractéristiques importantes (ex : texture de corrosion).

🔧 Étapes principales :


Renommage des fichiers

Supprime les accents et caractères spéciaux dans les noms de fichiers (ex : Capture d’écran → Capture_decran)
Évite les erreurs de lecture avec OpenCV



Chargement et nettoyage du dataset

Filtre uniquement les images (.jpg, .png, .jpeg)
Ignore les images non lisibles ou corrompues



Copie des images originales

Toutes les images valides sont copiées dans le dossier de sortie



Data augmentation (géométrique uniquement)

Applique des transformations légères :

flip horizontal / vertical
rotation faible
petite translation
crop léger + resize


Aucun changement de couleur ni zoom important



Génération jusqu’à 2000 images

Génère automatiquement de nouvelles images augmentées
S’arrête dès que le total atteint exactement 2000




✅ Objectif :

Créer un dataset plus grand, varié et robuste sans altérer les informations visuelles importantes (ex : corrosion chalconatronite).

In [11]:
import os
import cv2
import random
import unicodedata
from tqdm import tqdm
import albumentations as A

# =========================
# CONFIG
# =========================
input_dir = "C:/Users/CD44/Desktop/Traitement_images/DONNEES/tt_vrai_positif"
output_dir = "C:/Users/CD44/Desktop/Traitement_images/DONNEES/images_augmented_test_5"
target_size = 2000

os.makedirs(output_dir, exist_ok=True)

# =========================
# 1. RENOMMER (fix accents)
# =========================
def normalize_name(name):
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    name = name.replace(" ", "_")
    return name

for filename in os.listdir(input_dir):
    old_path = os.path.join(input_dir, filename)

    if not os.path.isfile(old_path):
        continue

    new_name = normalize_name(filename)
    new_path = os.path.join(input_dir, new_name)

    if old_path != new_path:
        base, ext = os.path.splitext(new_name)
        i = 1
        while os.path.exists(new_path):
            new_name = f"{base}_{i}{ext}"
            new_path = os.path.join(input_dir, new_name)
            i += 1

        os.rename(old_path, new_path)

print("✅ Renommage terminé")

# =========================
# 2. AUGMENTATION (SAFE CHALCO)
# =========================
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),

    # rotation légère
    A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.7),

    # translation légère (pas de zoom)
    A.ShiftScaleRotate(
        shift_limit=0.02,
        scale_limit=0.0,
        rotate_limit=0,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.5
    ),

    # ✅ CROP SAFE (remplace RandomCrop)
    A.RandomResizedCrop(
        size=(256, 256),
        scale=(0.8, 1.0),
        ratio=(1.0, 1.0),
        p=0.5
    )
])

# =========================
# 3. LOAD IMAGES
# =========================
images = [
    f for f in os.listdir(input_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# =========================
# 4. COPY ORIGINALS
# =========================
valid_images = []

for img_name in images:
    img_path = os.path.join(input_dir, img_name)
    img = cv2.imread(img_path)

    if img is None:
        print(f"⚠️ Image ignorée: {img_path}")
        continue

    valid_images.append(img_name)
    cv2.imwrite(os.path.join(output_dir, img_name), img)

current_count = len(valid_images)

# sécurité
if current_count >= target_size:
    print("✅ Dataset déjà >= target_size")
    exit()

# =========================
# 5. AUGMENTATION LOOP
# =========================
idx = 0
pbar = tqdm(total=target_size - current_count)

while current_count < target_size:
    img_name = random.choice(valid_images)
    img_path = os.path.join(input_dir, img_name)

    image = cv2.imread(img_path)
    if image is None:
        continue

    augmented = transform(image=image)
    aug_img = augmented["image"]

    new_name = f"aug_{idx}_{img_name}"
    cv2.imwrite(os.path.join(output_dir, new_name), aug_img)

    idx += 1
    current_count += 1
    pbar.update(1)

pbar.close()

print("✅ Augmentation terminée :", current_count, "images")


✅ Renommage terminé


100%|██████████| 889/889 [07:56<00:00,  1.87it/s]

✅ Augmentation terminée : 2000 images
